# Netflix Hit Proposal

Working notebook for the capstone deliverable: use real data from `netflix/data/*.csv` to (1) define what counts as a "hit", (2) find which pre-release-knowable features correlate with it, and (3) pitch an original series concept backed by that evidence.

This notebook runs on the **real** composite datasets (not the synthetic stand-in used in `netflix_review_concepts_capstone.ipynb`).

In [ ]:
import numpy as np
import pandas as pd

netflix = pd.read_csv("netflix/data/netflix.titles.composite.csv")
netflix.shape

## 1. Data completeness

Before defining anything, check how much of the composite dataset actually has each signal. `netflix_viewing_hours` and `netflix_weeks` come straight from Netflix's own Top 10 data and are complete for all 691 titles. The `tmdb_*` and `imdb_*` columns depend on a fuzzy-match join and are missing for a meaningful share of rows -- this is why they should stay out of the `hit` label itself (see Section 2).

In [ ]:
missing_pct = netflix.isna().mean().mul(100).round(1).rename("missing_pct")
missing_pct.to_frame()

## 2. Do viewing hours, votes, and ratings actually move together?

Log1p-transform the heavily skewed count-like columns (`viewing_hours`, `tmdb_popularity`, `tmdb_vote_count`, `imdb_numVotes`) before correlating -- their raw distributions span orders of magnitude and would otherwise be dominated by a handful of outliers (e.g. Stranger Things).

**Finding:** `netflix_viewing_hours` correlates strongly with `netflix_weeks` (0.70) but only weakly with `imdb_averageRating` (0.18) and barely at all with `imdb_numVotes` (0.08). Reach (how much something got watched) and reception (how well it was rated) are largely independent signals here -- which is exactly why they shouldn't be collapsed into one `hit` label. `viewing_hours` and `weeks` are the only two that clearly measure the same underlying thing (Netflix-internal popularity), so those are the two combined into `hit_score` below.

In [ ]:
log_cols = {
    "viewing_hours": np.log1p(netflix["netflix_viewing_hours"]),
    "weeks": np.log1p(netflix["netflix_weeks"]),
    "tmdb_popularity": np.log1p(netflix["tmdb_popularity"]),
    "tmdb_vote_average": netflix["tmdb_vote_average"],
    "tmdb_vote_count": np.log1p(netflix["tmdb_vote_count"]),
    "imdb_averageRating": netflix["imdb_averageRating"],
    "imdb_numVotes": np.log1p(netflix["imdb_numVotes"]),
}
corr_df = pd.DataFrame(log_cols)
corr_df.corr().round(2)

## 3. Defining `hit_score`

Decision: combine `netflix_viewing_hours` (total exposure) and `netflix_weeks` (durability -- did it stick around or spike and vanish) into one composite target. Ratings and vote counts stay out: they're weakly correlated with viewing hours (Section 2), would drop the usable sample from 691 to as few as 278 rows, and -- for the proposal use case -- don't exist yet for an unmade show anyway.

**Combination method:** z-score standardize each of `log1p(viewing_hours)` and `log1p(weeks)` independently, average them with equal weight, then min-max the result to `[0, 1]`.

Why equal weight and not a fitted weight: with exactly two already-standardized variables, PCA's first component is *always* the equal-weighted average, regardless of their correlation (the eigenvectors of `[[1, rho], [rho, 1]]` are always `(1,1)` and `(1,-1)`). There's also no ground-truth "hit" label available to fit a weight against via regression -- that would be circular, since `hit_score` is the thing being constructed. So equal weight isn't a shortcut; given two standardized inputs and no external criterion, it's what the data-driven approach reduces to.

In [ ]:
def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()

def minmax(s: pd.Series) -> pd.Series:
    return (s - s.min()) / (s.max() - s.min())

z_hours = zscore(np.log1p(netflix["netflix_viewing_hours"]))
z_weeks = zscore(np.log1p(netflix["netflix_weeks"]))

netflix["hit_score"] = minmax((z_hours + z_weeks) / 2)
netflix[["netflix_title", "netflix_viewing_hours", "netflix_weeks", "hit_score"]].sort_values(
    "hit_score", ascending=False
).head(10)

## 4. Sanity check

`hit_score` should reward both massive-but-brief spikes and modest-but-sustained runs, not just raw hours. Compare the ranking above to a plain `viewing_hours`-only ranking -- titles with high weeks but comparatively lower hours should move up; one-off spikes with very few weeks should move down relative to the hours-only ranking.

In [ ]:
hours_rank = netflix["netflix_viewing_hours"].rank(ascending=False)
hit_rank = netflix["hit_score"].rank(ascending=False)
netflix["rank_shift"] = hours_rank - hit_rank  # positive = moved up under hit_score
netflix[["netflix_title", "netflix_viewing_hours", "netflix_weeks", "hit_score", "rank_shift"]].sort_values(
    "rank_shift", ascending=False
).head(10)

## Next steps (not yet done)

- Feature engineering restricted to pre-release-knowable signals only: `genres`, `original_language`/`origin_country`, `episode_run_time` -- excluding anything that only exists after a title airs (ratings, vote counts, `number_of_seasons`/`number_of_episodes`, since renewal itself is a consequence of being a hit).
- Cast/crew historical track record (aggregate the *past* performance of attached writers/directors via the IMDb dataset) -- deferred for now.
- Fit the linear regression of `hit_score` on the feature set above; inspect coefficients and significance.
- Pick a Polti dramatic situation for the pitched show and write the proposal, using the regression coefficients as supporting evidence.